In [1]:
# === Google Colab 自動環境セットアップ ===
# ※ローカル環境では安全に無視され、Colab環境でのみ自動実行されます
import os, sys
repo_root = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.exists("../common") else os.path.abspath(os.getcwd())
if "google.colab" in sys.modules:
    print("⚡ Google Colab 環境を検出しました。リポジトリのセットアップを開始します...")
    !git clone https://github.com/miyayudai/study-mikanos.git /content/study-mikanos 2>/dev/null || (cd /content/study-mikanos && git pull)
    %cd /content/study-mikanos
    !pip install -q -r requirements.txt
    !pip install -q -e .
    repo_root = "/content/study-mikanos"
    sys.path.insert(0, repo_root)
    print("✅ 環境セットアップが完了しました。作業ディレクトリ:", os.getcwd())
else:
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    print("💻 ローカル・開発環境で実行中。リポジトリルート:", repo_root)


💻 ローカル・開発環境で実行中。リポジトリルート: /home/student/Documents/GitHub/study-mikanos


# 6.3 PCIデバイスの探索 (`Scanning PCI Configuration Space`)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/miyayudai/study-mikanos/blob/main/6/6.3_Scanning_PCI_Configuration_Space.ipynb)
![Language](https://img.shields.io/badge/Language-C%2B%2B20%20%7C%20Python%203.11%2B-blue.svg)
![Target](https://img.shields.io/badge/Target-x86__64%20%7C%20PCI%20Local%20Bus-orange.svg)
![Chapter](https://img.shields.io/badge/Chapter-6.3%20(PCI%20Device%20Discovery)-green.svg)

---

## 📖 節の概要と学習目標

前節（**第6章 6.2節**）では、USB 3.0 および最新のホストコントローラ規格である **xHCI（*eXtensible Host Controller Interface*）** の内部メモリ構造（DCBAA、Device Context、Transfer/Command/Event リング、16バイトTRB、Cycle Bit ロックフリー同期数理）を仕様書の根本理論から学びました。

しかし、前節までの知識だけでは、実際に OS カーネルから USB マウスを制御することはできません。なぜなら、**「PC の広大なハードウェア空間のどこに xHCI ホストコントローラが物理的に接続されているのか？」** をまだ特定できていないからです。

現代の PC アーキテクチャにおいて、グラフィックカード、ネットワークコントローラ、ストレージ（NVMe / SATA）、そして USB ホストコントローラなどの主要周辺機器は、すべて **PCI（*Peripheral Component Interconnect*）** またはその高速後継規格である **PCI Express (PCIe)** バスを介して CPU に接続されています。

本節（**第6章 6.3節 / `osbook_day06b`**）では、OS カーネルがマシン上のすべての PCI デバイスを自律的に検出し、目標とする xHCI コントローラを特定するための **「PCI コンフィギュレーション空間の探索（Scanning PCI Configuration Space）」** を完全導出・実装します：

1. **周辺機器接続の歴史と PCI バストポロジー**: レガシー ISA のジャンパ設定から PCI プラグ・アンド・プレイへの進化、およびホストブリッジ・PCI-to-PCI ブリッジ・バス階層ツリー
2. **ポートマップド I/O (PMIO) と x86-64 `in` / `out` 命令**: メモリ空間（MMIO）と I/O ポート空間（PMIO）の違い、および System V AMD64 ABI に準拠したアセンブリ関数（`IoIn32`, `IoOut32`）の実装
3. **PCI Configuration Mechanism #1 の数理導出**: I/O ポート `0x0CF8` (`CONFIG_ADDRESS`) と `0x0CFC` (`CONFIG_DATA`) による 32 ビット単位のレジスタアクセス数理
4. **標準コンフィギュレーション空間ヘッダ（Type 00h & Type 01h）**: Vendor ID (`0xFFFF` 未接続判定)、Device ID、Class Code、Header Type（多重ファンクション判定）の構造
5. **再帰的 PCI バス全探索アルゴリズム (`ScanAllBus`)**: バス 0〜255、デバイス 0〜31、ファンクション 0〜7 の走査ロジックと、PCI-to-PCI ブリッジ検出時のセカンダリバス再帰探索
6. **xHCI USB 3.0 コントローラの特定数理**: Class Code `0x0C0330xx`（Serial Bus: USB, Prog IF: xHCI）によるフィルタリングと QEMU デバイスツリーの完全再現


## 1. なぜ PCI デバイス探索が必要なのか？（動機と周辺機器バスの進化）

### 1.1 ISA バスの暗黒時代とハードウェア競合の課題
1980年代の PC/AT 互換機で主流だった **ISA（*Industry Standard Architecture*）** バスでは、周辺機器をマザーボードに増設する際、各拡張カード上の物理的な **ジャンパピン（Jumper Pin）** や DIP スイッチを人間が手作業で切り替え、I/O ポート番号や割り込み番号（IRQ）を手動設定していました。

```
[ISA 時代の問題点]
1. リソース衝突: 2枚のカードが同じ IRQ や I/O アドレスを要求するとシステムがクラッシュ
2. アドレス固定化: デバイスの位置がハードコーディングされ、OS はプラグ・アンド・プレイが不可能
3. 狭い帯域幅: 16ビット幅・8MHz（最大 16 MB/s）でグラフィックや高速ストレージに対応不能
```

### 1.2 PCI バスの誕生とプラグ・アンド・プレイ (PnP)
この課題を根本解決するため、Intel は 1992 年に **PCI（*Peripheral Component Interconnect*）** 仕様を策定しました。PCI の最大の特徴は、**「すべての周辺機器が自己記述的なコンフィギュレーション空間（Configuration Space）を持つ」** という点です。

OS は起動時にハードウェアに対して「あなたは誰ですか？どのようなリソースを必要としていますか？」と問い合わせるだけで、ジャンパピンなしでデバイスを自動検出し、競合のないアドレスを動的に割り当てることができるようになりました。

### 1.3 PCI から PCI Express (PCIe) への進化と後方互換性
現在ではパラレル伝送の PCI バスに代わり、高速シリアルポイントツーポイントリンクの **PCI Express (PCIe)** が主流です。しかし、**「ソフトウェアから見た論理階層（バス番号 0〜255、デバイス番号 0〜31、ファンクション番号 0〜7、256バイトのコンフィギュレーション空間）」は PCI 時代と完全に同一の後方互換性が保たれています**。

したがって、MikanOS で学ぶ PCI 探索ロジックは、最新の Intel Core や AMD Ryzen プロセッサ上の PCIe デバイス探索でもそのまま通用する普遍の基本原理です。

## 2. ポートマップド I/O (PMIO) と x86-64 `in` / `out` 命令

### 2.1 メモリ空間 (MMIO) と I/O ポート空間 (PMIO) の相違
x86-64 CPU には、2 つの独立したアドレス空間が存在します：

1. **メモリ空間 (Memory Space / MMIO: *Memory-Mapped I/O*)**:
   - 通常の DRAM や UEFI フレームバッファ（VRAM）がマップされる広大な 64 ビット空間（$2^{64}$ バイト）。
   - ポインタ間接参照（C++ の `*ptr = val` や `mov [rax], rbx`）で自由に読み書き可能。
2. **I/O ポート空間 (I/O Port Space / PMIO: *Port-Mapped I/O*)**:
   - x86 CPU 独自のアドレス空間であり、**16 ビット幅（0x0000 〜 0xFFFF = 65,536 個のポート）** に限定される。
   - 通常のポインタや `mov` 命令では絶対にアクセスできず、専用の機械語命令 **`in`** および **`out`** を使用する。

| 項目 | メモリ空間 (MMIO) | I/O ポート空間 (PMIO) |
|---|---|---|
| **アドレス幅** | 64-bit (0x0000000000000000 〜 0xFFFFFFFFFFFFFFFF) | 16-bit (0x0000 〜 0xFFFF, 64 KiB) |
| **アクセス命令** | `mov`, `memcpy`, 通常の C++ ポインタ読み書き | `in`, `out` 機械語命令のみ |
| **C/C++ 直接記述** | 可能 (`uint32_t* p = ...; *p = val;`) | **不可**（インラインアセンブリまたは独立関数が必要） |
| **用途** | フレームバッファ, xHCI MMIO レジスタ, DRAM | PCI 設定空間アクセス, レガシータイマ, シリアルポート |

### 2.2 System V AMD64 ABI に準拠したアセンブリ関数 (`IoIn32`, `IoOut32`)
C++ からは I/O ポート命令を直接発行できないため、MikanOS では独立したアセンブリファイル `kernel/asmfunc.asm` にラッパー関数を定義し、ヘッダファイル `kernel/asmfunc.h` で宣言します。

**System V AMD64 ABI（*Application Binary Interface*）の引数レジスタ規約**:
- 第1引数: `RDI`（下位16ビットは `DI`）
- 第2引数: `RSI`（下位32ビットは `ESI`）
- 戻り値: `RAX`（下位32ビットは `EAX`）

```nasm
; kernel/asmfunc.asm
bits 64
section .text

global IoOut32  ; void IoOut32(uint16_t addr, uint32_t data);
IoOut32:
    mov dx, di    ; 第1引数 addr (16-bit) を dx レジスタに格納
    mov eax, esi  ; 第2引数 data (32-bit) を eax レジスタに格納
    out dx, eax   ; I/Oポート dx に 32ビット値 eax を出力！
    ret

global IoIn32   ; uint32_t IoIn32(uint16_t addr);
IoIn32:
    mov dx, di    ; 第1引数 addr (16-bit) を dx レジスタに格納
    in eax, dx    ; I/Oポート dx から 32ビット値を読み取り eax に格納！
    ret           ; eax の値が戻り値として C++ 呼び出し元に返る
```

```cpp
// kernel/asmfunc.h
#pragma once
#include <stdint.h>

extern "C" {
  void IoOut32(uint16_t addr, uint32_t data);
  uint32_t IoIn32(uint16_t addr);
}
```

## 3. PCI Configuration Mechanism #1 の数理と `CONFIG_ADDRESS` 導出

### 3.1 2つの専用 I/O ポート (`0x0CF8` と `0x0CFC`)
x86 PC において、CPU が PCI コンフィギュレーション空間（各デバイス 256 バイト）にアクセスするために用意されている標準的な仕組みが **PCI Configuration Mechanism #1** です。この仕組みでは、以下の 2 つの 32 ビット I/O ポートを用います：

1. **`0x0CF8` : `CONFIG_ADDRESS` (設定アドレスレジスタ)**:
   - アクセスしたい PCI デバイスの「バス番号」「デバイス番号」「ファンクション番号」「レジスタオフセット」を指定する 32 ビットレジスタ。
2. **`0x0CFC` : `CONFIG_DATA` (設定データレジスタ)**:
   - `CONFIG_ADDRESS` で指定されたレジスタの内容がそのまま現れる 32 ビットのデータ窓口。このポートに対して `IoIn32` / `IoOut32` を行うことで、実際の読み書きがハードウェア的に実行される。

### 3.2 `CONFIG_ADDRESS` (32ビット) のビットフィールド数理導出
`CONFIG_ADDRESS` レジスタ（32 ビット）のビットフィールドは PCI 仕様で以下のように厳密に定義されています：

| ビット範囲 | フィールド名 | 幅 | 説明・値域 |
|---|---|:---:|---|
| **Bit 31** | **Enable Bit** | 1 bit | コンフィギュレーションアクセスの有効化フラグ（**`1` = 有効**, `0` = 無効） |
| **Bits 30:24** | **Reserved** | 7 bits | 予約領域（常に **`0`**） |
| **Bits 23:16** | **Bus Number** | 8 bits | PCI バス番号（**0 〜 255**） |
| **Bits 15:11** | **Device Number** | 5 bits | デバイス番号（**0 〜 31**） |
| **Bits 10:8** | **Function Number** | 3 bits | ファンクション番号（**0 〜 7**） |
| **Bits 7:2** | **Register Offset** | 6 bits | レジスタオフセット（4バイト境界、**0x00 〜 0xFC**） |
| **Bits 1:0** | **Offset Alignment** | 2 bits | 常に **`00b`**（32ビット DWORD アライメント固定） |

### 3.3 数式導出と C++20 実装
上記のビット配置から、任意の `(bus, device, function, reg_addr)` に対する 32 ビット整数 `CONFIG_ADDRESS` の計算式は以下の論理和（OR）とビットシフト（SHL）によって一意に導出されます：

$$\mathbf{CONFIG\_ADDRESS} = (1 \ll 31) \mid (\text{bus} \ll 16) \mid (\text{device} \ll 11) \mid (\text{function} \ll 8) \mid (\text{reg\_addr} \ \& \ \text{0xFC})$$

```cpp
uint32_t MakeAddress(uint8_t bus, uint8_t device,
                     uint8_t function, uint8_t reg_addr) {
  auto shl = [](uint32_t x, unsigned int bits) {
    return x << bits;
  };

  return shl(1, 31)  // enable bit (Bit 31)
      | shl(bus, 16)
      | shl(device, 11)
      | shl(function, 8)
      | (reg_addr & 0xfcu); // 下位2ビットをゼロマスクして4バイトアライメント保証
}
```

> [!NOTE]
> `CONFIG_DATA` (`0x0CFC`) はハードウェア仕様上 **32ビット単位でのみアクセス可能** です。8ビット値（Header Type等）や16ビット値（Vendor ID等）を取得したい場合は、まず 32 ビット一括で読み出した後に、適切なビットシフトとマスク演算を行う必要があります。

## 4. PCI コンフィギュレーション空間標準ヘッダ（Type 00h & Type 01h）

各 PCI デバイス（ファンクション）は **256 バイト** のコンフィギュレーション空間を持っています。その先頭 64 バイト（16 DWORD）は全デバイス共通の「定義済みヘッダ（Predefined Header）」となっており、デバイスの種類によって 2 種類のレイアウトが存在します。

### 4.1 全ヘッダ共通の先頭 16 バイト
すべての PCI デバイスにおいて、先頭 16 バイト（Offset 0x00 〜 0x0F）の構造は完全に共通です：

1. **Offset 0x00 : Vendor ID (下位16ビット) / Device ID (上位16ビット)**
   - **Vendor ID**: PCI-SIG がメーカーごとに割り当てる 16 ビット固有番号（例: Intel は `0x8086`, QEMU/Red Hat は `0x1B36`, NVIDIA は `0x10DE`）。
   - **★存在判定の鉄則**: 指定した `(bus, device, function)` にデバイスが存在しない場合、ハードウェアはデータバスをプルアップ（全ビット1）して **`0xFFFF`** を返します。したがって、**`ReadVendorId() == 0xFFFF` であれば、そのデバイスは未実装（スキャン対象外）** と即座に判定できます！
2. **Offset 0x08 : Revision ID (下位8ビット) / Class Code (上位24ビット)**
   - デバイスの機能種別を表す極めて重要な 24 ビット識別子。
   - `[31:24]` : **Base Class**（大分類。例: `0x0C` = Serial Bus Controller, `0x06` = Bridge Device, `0x03` = Display）
   - `[23:16]` : **Sub Class**（中分類。例: `0x03` = USB Controller, `0x04` = PCI-to-PCI Bridge）
   - `[15:8]`  : **Programming Interface (Prog IF)**（詳細仕様。例: `0x30` = xHCI, `0x20` = EHCI, `0x00` = UHCI）
   - `[7:0]`   : **Revision ID**（リビジョン番号）
3. **Offset 0x0C : Header Type (Bits 23:16 of DWORD 3)**
   - **Bit 7 (0x80) : Multi-Function Flag**:
     - `0`: **単一ファンクションデバイス**（Function 0 のみ存在。Function 1〜7 は走査不要！）
     - `1`: **多重ファンクションデバイス**（Function 1〜7 にも別機能のデバイスが存在する可能性があるため、走査が必要！）
   - **Bits 6:0 : Header Layout**:
     - `0x00` : **Header Type 00h**（一般的なエンドポイントデバイス: NIC、GPU、USBコントローラ等）
     - `0x01` : **Header Type 01h**（**PCI-to-PCI ブリッジ**: 階層的バスを中継するブリッジ）

### 4.2 Header Type 01h (PCI-to-PCI ブリッジ) と再帰的バス探索
Header Type が `0x01` のデバイスは、バスとバスを接続する **PCI-to-PCI ブリッジ** です。
ブリッジの Offset 0x18 には以下の重要なバス番号レジスタが存在します：
- `Bits 7:0`   : **Primary Bus Number**（親バス番号）
- `Bits 15:8`  : **Secondary Bus Number**（**セカンダリバス番号 = ブリッジの先に接続された子バスの番号**）
- `Bits 23:16` : **Subordinate Bus Number**（配下に存在する最大バス番号）

OS は PCI-to-PCI ブリッジを発見した際、**Secondary Bus Number を読み取り、その子バスに対して再帰的に探索（`ScanBus(secondary_bus)`）を実行する** ことで、マザーボード上のすべての階層下デバイスを漏れなく発見できます。

In [2]:
# === Python による PCIAddress ビット演算導出と PCI 設定空間読み取りシミュレーション ===
from common import PCIAddress, PCIDeviceSim, PCIConfigSpaceSimulator

# 1. MakeAddress ビット演算の数理検証
# 例: Bus 0, Dev 4, Func 0, Reg 0x00 (xHCI Vendor/Device ID)
addr_xhci = PCIAddress(bus=0, device=4, function=0, reg_addr=0x00)
raw_uint32 = addr_xhci.to_uint32()

print("=== PCI Address 計算数理 ===")
print(f"Target: Bus={addr_xhci.bus}, Device={addr_xhci.device}, Func={addr_xhci.function}, Reg=0x{addr_xhci.reg_addr:02x}")
print(f"CONFIG_ADDRESS (Hex): 0x{raw_uint32:08X}")
print(f"CONFIG_ADDRESS (Bin): {bin(raw_uint32)}")
print(f"  - Bit 31 (Enable) : {(raw_uint32 >> 31) & 1}")
print(f"  - Bits 23:16 (Bus): {(raw_uint32 >> 16) & 0xFF}")
print(f"  - Bits 15:11 (Dev): {(raw_uint32 >> 11) & 0x1F}")
print(f"  - Bits 10:8 (Func): {(raw_uint32 >> 8) & 0x07}")
print(f"  - Bits 7:2 (Reg)  : {(raw_uint32 & 0xFC):02x}")

# 2. QEMU 標準トポロジーの PCI 設定空間エミュレータ初期化
sim = PCIConfigSpaceSimulator.create_qemu_standard_pci_space()

# 各種レジスタの読み出し検証
vid_host = sim.read_vendor_id(0, 0, 0)
did_host = sim.read_device_id(0, 0, 0)
head_host = sim.read_header_type(0, 0, 0)

vid_xhci = sim.read_vendor_id(0, 4, 0)
did_xhci = sim.read_device_id(0, 4, 0)
cc_xhci = sim.read_class_code(0, 4, 0)

print("\n=== PCI レジスタ読み取り検証 ===")
print(f"[0.0.0 Host Bridge] Vendor: 0x{vid_host:04X}, Device: 0x{did_host:04X}, HeaderType: 0x{head_host:02X}")
print(f"[0.4.0 xHCI USB 3]  Vendor: 0x{vid_xhci:04X}, Device: 0x{did_xhci:04X}, ClassCode: 0x{cc_xhci:08X}")
assert vid_xhci == 0x1B36 and did_xhci == 0x000D, "xHCI Vendor/Device ID must match QEMU specification"
print("✅ Python による PCI CONFIG 読み取りシミュレーション正常終了！")


=== PCI Address 計算数理 ===
Target: Bus=0, Device=4, Func=0, Reg=0x00
CONFIG_ADDRESS (Hex): 0x80002000
CONFIG_ADDRESS (Bin): 0b10000000000000000010000000000000
  - Bit 31 (Enable) : 1
  - Bits 23:16 (Bus): 0
  - Bits 15:11 (Dev): 4
  - Bits 10:8 (Func): 0
  - Bits 7:2 (Reg)  : 00

=== PCI レジスタ読み取り検証 ===
[0.0.0 Host Bridge] Vendor: 0x8086, Device: 0x1237, HeaderType: 0x00
[0.4.0 xHCI USB 3]  Vendor: 0x1B36, Device: 0x000D, ClassCode: 0x0C033001
✅ Python による PCI CONFIG 読み取りシミュレーション正常終了！


## 5. 高解像度アーキテクチャ図版（Figure 6.05 & Figure 6.06）

PCI デバイス探索の核心を直感的に理解できるよう、本リポジトリでは 2 つの高解像度図版（Figure 6.05 および Figure 6.06）を完全再現しています：

- **Figure 6.05 : PCI CONFIG_ADDRESS (0x0CF8) レジスタ構造と Type 00h / 01h ヘッダレイアウト詳細マップ**
  - 上段: 32 ビット幅 `CONFIG_ADDRESS` レジスタの各フィールド（Enable, Reserved, Bus, Device, Function, Register Offset）のビット配置とアドレス生成数理
  - 下段左: 一般デバイス向け **Header Type 00h** の 64 バイトレジスタマップ（BAR0〜BAR5、割り込みピン等）
  - 下段右: ブリッジ向け **Header Type 01h** の 64 バイトレジスタマップ（Primary, Secondary, Subordinate バス番号レジスタ等）
- **Figure 6.06 : PCI バストポロジー階層図と ScanAllBus 再帰的探索アルゴリズムの制御フロー**
  - 左側: QEMU 仮想環境における PCI バストポロジー（CPU → Host Bridge Bus 0 → Dev 0〜5 → PCI Bridge → Secondary Bus 1 → Audio）
  - 右側: `ScanAllBus()` から `ScanBus()`, `ScanDevice()`, `ScanFunction()` を経てブリッジ検出時に再帰呼び出しを行う完全な状態遷移フローチャート

![Figure 6.05](result/fig6_05_pci_config_address_and_header_layout.png)

![Figure 6.06](result/fig6_06_pci_bus_tree_and_recursive_scan_algorithm.png)

In [3]:
# === Figure 6.05 および Figure 6.06 の生成と検証 ===
from pathlib import Path
from common import (
    plot_fig6_05_pci_config_address_and_header_layout,
    plot_fig6_06_pci_bus_tree_and_recursive_scan_algorithm
)

fig1_path = "result/fig6_05_pci_config_address_and_header_layout.png"
fig2_path = "result/fig6_06_pci_bus_tree_and_recursive_scan_algorithm.png"

# 図版の生成（すでに存在する場合はそのままロード）
p1 = plot_fig6_05_pci_config_address_and_header_layout(fig1_path)
p2 = plot_fig6_06_pci_bus_tree_and_recursive_scan_algorithm(fig2_path)

print(f"✅ 生成完了: {p1} ({Path(p1).stat().st_size:,} bytes)")
print(f"✅ 生成完了: {p2} ({Path(p2).stat().st_size:,} bytes)")
print("Figure 6.05 & 6.06 は result/ ディレクトリに高解像度 PNG (300 DPI) として保存され、上部の Markdown セルに埋め込まれています。")


✅ 生成完了: result/fig6_05_pci_config_address_and_header_layout.png (832,454 bytes)
✅ 生成完了: result/fig6_06_pci_bus_tree_and_recursive_scan_algorithm.png (702,304 bytes)
Figure 6.05 & 6.06 は result/ ディレクトリに高解像度 PNG (300 DPI) として保存され、上部の Markdown セルに埋め込まれています。


## 6. 再帰的 PCI バス探索アルゴリズム (`ScanAllBus` の設計と完全導出)

MikanOS の PCI スキャナ（`kernel/pci.cpp`）は、関数の責務を明確に分離した 4 段階の階層構造で実装されています。

```
ScanAllBus()
  └── [Host Bridge 0.0.0 が単一ファンクションなら] ──> ScanBus(0)
  └── [Host Bridge 0.0.0 が多重ファンクションなら] ──> ScanBus(function 1..7)
        └── ScanBus(bus): デバイス 0..31 を走査 (Vendor ID != 0xFFFF)
              └── ScanDevice(bus, device): Function 0 を走査
                    └── [多重ファンクションなら] ──> Function 1..7 を走査
                          └── ScanFunction(bus, device, function)
                                ├── devices[num_device++] に登録
                                └── [PCI-to-PCI ブリッジなら] ──> 再帰呼び出し: ScanBus(secondary_bus) ★
```

### 6.1 `ScanFunction()` の実装
単一のファンクションを `devices` 配列に登録し、もし PCI-to-PCI ブリッジであれば Secondary Bus 番号を取得して `ScanBus()` を再帰呼び出しします：

```cpp
Error ScanFunction(uint8_t bus, uint8_t device, uint8_t function) {
  auto header_type = ReadHeaderType(bus, device, function);
  if (auto err = AddDevice(bus, device, function, header_type)) {
    return err; // devices 配列が満杯 (32個) なら Error::kFull を返す
  }

  auto class_code = ReadClassCode(bus, device, function);
  uint8_t base = (class_code >> 24) & 0xffu;
  uint8_t sub = (class_code >> 16) & 0xffu;

  if (base == 0x06u && sub == 0x04u) {
    // standard PCI-PCI bridge (Class Code = 0x0604)
    auto bus_numbers = ReadBusNumbers(bus, device, function);
    uint8_t secondary_bus = (bus_numbers >> 8) & 0xffu;
    return ScanBus(secondary_bus); // 再帰呼び出しで子バスを走査！
  }

  return Error::kSuccess;
}
```

### 6.2 `ScanDevice()` の実装
デバイス内の各ファンクションを走査します。まず Function 0 を調べ、**単一ファンクションデバイスであれば Function 1〜7 の走査を完全に省略** することで、バス探索時間を大幅に短縮します：

```cpp
Error ScanDevice(uint8_t bus, uint8_t device) {
  if (auto err = ScanFunction(bus, device, 0)) {
    return err;
  }
  if (IsSingleFunctionDevice(ReadHeaderType(bus, device, 0))) {
    return Error::kSuccess; // 単一ファンクションなら即座に終了！
  }

  for (uint8_t function = 1; function < 8; ++function) {
    if (ReadVendorId(bus, device, function) == 0xffffu) {
      continue; // 未接続ファンクションはスキップ
    }
    if (auto err = ScanFunction(bus, device, function)) {
      return err;
    }
  }
  return Error::kSuccess;
}
```

### 6.3 `ScanBus()` と `ScanAllBus()` の実装
`ScanBus()` は指定バス上の 32 個のデバイス（0〜31）を走査します。
`ScanAllBus()` はシステム全体の起点となる関数であり、**ホストブリッジ（Bus 0, Device 0, Function 0）** の Header Type を確認して探索を開始します：

```cpp
Error ScanBus(uint8_t bus) {
  for (uint8_t device = 0; device < 32; ++device) {
    if (ReadVendorId(bus, device, 0) == 0xffffu) {
      continue; // デバイス自体が存在しなければスキップ
    }
    if (auto err = ScanDevice(bus, device)) {
      return err;
    }
  }
  return Error::kSuccess;
}

Error ScanAllBus() {
  num_device = 0;

  auto header_type = ReadHeaderType(0, 0, 0);
  if (IsSingleFunctionDevice(header_type)) {
    return ScanBus(0); // 通常の単一ホストブリッジ構成
  }

  for (uint8_t function = 1; function < 8; ++function) {
    if (ReadVendorId(0, 0, function) == 0xffffu) {
      continue;
    }
    if (auto err = ScanBus(function)) {
      return err;
    }
  }
  return Error::kSuccess;
}
```

## 7. xHCI USB 3.0 ホストコントローラの特定数理

`ScanAllBus()` によって `devices` 配列にすべての PCI デバイスが格納された後、カーネルは **「どれが xHCI USB ホストコントローラなのか？」** を判定します。

### 7.1 Class Code によるピンポイント特定
PCI 仕様において、USB コントローラは以下の Class Code で分類されています：

- **Base Class = `0x0C`** : *Serial Bus Controller*（シリアルバスコントローラ）
- **Sub Class = `0x03`** : *Universal Serial Bus (USB)*（USBコントローラ）
- **Programming Interface (Prog IF)**:
  - `0x00` : **UHCI** (*Universal Host Controller Interface* - USB 1.1 Intel)
  - `0x10` : **OHCI** (*Open Host Controller Interface* - USB 1.1 Compaq/Microsoft)
  - `0x20` : **EHCI** (*Enhanced Host Controller Interface* - USB 2.0 High-Speed)
  - **`0x30`** : **xHCI (*eXtensible Host Controller Interface* - USB 3.0 SuperSpeed)** ★

### 7.2 ビットマスク数式と C++ 判定コード
32 ビットレジスタとして読み出される Class Code は `[31:24] Base`, `[23:16] Sub`, `[15:8] Prog IF`, `[7:0] Revision` です。したがって、リビジョン番号を無視して xHCI を判定する数式は以下のようになります：

$$\mathbf{IsXHCI}(\text{class\_code}) \iff (\text{class\_code} \gg 8) == \text{0x0C0330}$$

```cpp
// xHCI コントローラの探索ループ (次節 6.4節 で使用)
pci::Device* xhc_dev = nullptr;
for (int i = 0; i < pci::num_device; ++i) {
  auto class_code = pci::ReadClassCode(pci::devices[i].bus,
                                       pci::devices[i].device,
                                       pci::devices[i].function);
  if ((class_code >> 8) == 0x0c0330u) {
    xhc_dev = &pci::devices[i];
    break; // xHCI コントローラを発見！
  }
}
```

## 8. C++20 による PCI スキャナの対話型コンパイル & 検証

本リポジトリでは、MikanOS の `kernel/pci.cpp`, `kernel/pci.hpp`, `kernel/error.hpp` と全く同一のロジックをホスト環境でコンパイル・単体テストできる C++20 スタンドアロンシミュレータ（`src/ch06/pci_scan_sim.cpp`）を提供しています。

このシミュレータは、メモリ上に仮想的な PCI 空間と I/O ポート `0x0CF8` / `0x0CFC` の入出力を再現し、以下の項目を厳密に検証します：
1. `MakeAddress()` のビット配置・アライメントマスクの正確性
2. 未実装バスでの 0 件検出挙動
3. Header Type bit 7（単一/多重ファンクション）に応じた走査の切り替え
4. PCI-to-PCI ブリッジを検出した際のセカンダリバス（Bus 1）への再帰探索
5. QEMU 標準トポロジーにおける xHCI USB 3.0 コントローラの自動検出

In [4]:
# === C++20 スタンドアロン PCI スキャンシミュレータの実行 ===
import subprocess
from pathlib import Path

bin_path = Path(repo_root) / "build" / "bin" / "ch06_pci_scan_sim"
res = subprocess.run([str(bin_path), "--test"], capture_output=True, text=True, check=True)
print(res.stdout)


Running Chapter 6.3 PCI Configuration & Bus Scan Standalone Tests...
[PASS] TestMakeAddress: Bitfield layout and masking verified.
[PASS] TestEmptyBusScan: 0 devices detected on unpopulated bus.
[PASS] TestSingleVsMultiFunctionScan: Header Type bit 7 distinction verified.

=== Discovered PCI Devices (MikanOS ScanAllBus Simulation) ===
Status: kSuccess | Total Devices: 9
0.0.0: vend 0x8086, dev 0x1237, class 0x06000002, head 0x00
0.1.0: vend 0x8086, dev 0x7000, class 0x06010000, head 0x80
0.1.1: vend 0x8086, dev 0x7010, class 0x01018000, head 0x00
0.1.3: vend 0x8086, dev 0x7113, class 0x06800000, head 0x00
0.2.0: vend 0x1234, dev 0x1111, class 0x03000000, head 0x00
0.3.0: vend 0x8086, dev 0x100e, class 0x02000000, head 0x00
0.4.0: vend 0x1b36, dev 0x000d, class 0x0c033001, head 0x00 <-- [xHCI USB 3.0 Controller FOUND!]
0.5.0: vend 0x1b36, dev 0x0001, class 0x06040000, head 0x01 <-- [PCI-to-PCI Bridge]
1.1.0: vend 0x8086, dev 0x2415, class 0x04010000, head 0x00
[PASS] TestQEMUStandardSca

## 9. Python による QEMU PCI デバイスツリーの探索実演

MikanOS の `day06b` カーネル（`kernel/main.cpp`）では、起動時に `pci::ScanAllBus()` を呼び出し、発見されたすべてのデバイスを画面上のコンソールに以下のように出力します：

```
ScanAllBus: kSuccess
0.0.0: vend 8086, class 06000002, head 00
0.1.0: vend 8086, class 06010000, head 80
0.1.1: vend 8086, class 01018000, head 00
0.1.3: vend 8086, class 06800000, head 00
0.2.0: vend 1234, class 03000000, head 00
0.3.0: vend 8086, class 02000000, head 00
0.4.0: vend 1b36, class 0c033001, head 00
0.5.0: vend 1b36, class 06040000, head 01
1.1.0: vend 8086, class 04010000, head 00
```

Python シミュレータを用いて、この探索ログと xHCI 特定シーケンスを忠実に再現してみましょう。

In [5]:
# === Python による QEMU PCI デバイスツリー走査と MikanOS コンソール出力の再現 ===
from common import PCIConfigSpaceSimulator

sim = PCIConfigSpaceSimulator.create_qemu_standard_pci_space()
devices, logs = sim.scan_all_bus()

print("=== MikanOS kernel/main.cpp 画面出力の完全再現 ===")
print("ScanAllBus: kSuccess")
for dev in devices:
    print(f"{dev.bus}.{dev.device}.{dev.function}: vend {dev.vendor_id:04x}, class {dev.class_code:08x}, head {dev.header_type:02x}")

print("\n=== 発見されたデバイスの詳細解析 ===")
xhci_target = None
for dev in devices:
    desc = "一般デバイス"
    if dev.is_xhci:
        desc = "★ xHCI USB 3.0 コントローラ [目標デバイス]"
        xhci_target = dev
    elif dev.is_pci_bridge:
        desc = f"PCI-to-PCI ブリッジ (Secondary Bus = {dev.secondary_bus})"
    elif dev.base_class == 0x06 and dev.sub_class == 0x00:
        desc = "ホストブリッジ (Intel 82440FX)"
    elif dev.base_class == 0x06 and dev.sub_class == 0x01:
        desc = "ISA ブリッジ (Intel PIIX3)"
    elif dev.base_class == 0x03:
        desc = "VGA ディスプレイアダプタ"
    elif dev.base_class == 0x02:
        desc = "イーサネットコントローラ (Intel e1000)"
    elif dev.base_class == 0x04:
        desc = "AC'97 オーディオコントローラ (二次バス上)"

    print(f"[{dev.bus}.{dev.device}.{dev.function}] {desc} (Vendor: 0x{dev.vendor_id:04X}, Dev: 0x{dev.device_id:04X})")

print(f"\n🎯 目標コントローラの特定結果:")
assert xhci_target is not None, "xHCI コントローラが発見されなければなりません"
print(f"   -> PCI アドレス : Bus {xhci_target.bus}, Device {xhci_target.device}, Function {xhci_target.function}")
print(f"   -> Class Code  : 0x{xhci_target.class_code:08X} (Base=0x{xhci_target.base_class:02X}, Sub=0x{xhci_target.sub_class:02X}, ProgIF=0x{xhci_target.prog_if:02X})")
print("✅ QEMU トポロジー探索と xHCI 特定の完全検証に成功しました！")


=== MikanOS kernel/main.cpp 画面出力の完全再現 ===
ScanAllBus: kSuccess
0.0.0: vend 8086, class 06000002, head 00
0.1.0: vend 8086, class 06010000, head 80
0.1.1: vend 8086, class 01018000, head 00
0.1.3: vend 8086, class 06800000, head 00
0.2.0: vend 1234, class 03000000, head 00
0.3.0: vend 8086, class 02000000, head 00
0.4.0: vend 1b36, class 0c033001, head 00
0.5.0: vend 1b36, class 06040000, head 01
1.1.0: vend 8086, class 04010000, head 00

=== 発見されたデバイスの詳細解析 ===
[0.0.0] ホストブリッジ (Intel 82440FX) (Vendor: 0x8086, Dev: 0x1237)
[0.1.0] ISA ブリッジ (Intel PIIX3) (Vendor: 0x8086, Dev: 0x7000)
[0.1.1] 一般デバイス (Vendor: 0x8086, Dev: 0x7010)
[0.1.3] 一般デバイス (Vendor: 0x8086, Dev: 0x7113)
[0.2.0] VGA ディスプレイアダプタ (Vendor: 0x1234, Dev: 0x1111)
[0.3.0] イーサネットコントローラ (Intel e1000) (Vendor: 0x8086, Dev: 0x100E)
[0.4.0] ★ xHCI USB 3.0 コントローラ [目標デバイス] (Vendor: 0x1B36, Dev: 0x000D)
[0.5.0] PCI-to-PCI ブリッジ (Secondary Bus = 1) (Vendor: 0x1B36, Dev: 0x0001)
[1.1.0] AC'97 オーディオコントローラ (二次バス上) (Vendor: 0x8086, Dev: 0x241

## 10. コラム: レガシー I/O ポートアクセス (Mechanism #1) と PCIe ECAM (MMIO) の違い

### 10.1 なぜ MikanOS では古い I/O ポート機構を使うのか？
PCI の設定空間アクセス機構には、本節で実装した **Configuration Mechanism #1（I/O ポート 0xCF8 / 0xCFC）** のほかに、PCIe 世代の標準である **ECAM（*Enhanced Configuration Access Mechanism*）** が存在します。

なぜ最新の MikanOS であえてレガシーな Mechanism #1 を採用するのでしょうか？

1. **初期化の依存関係ゼロ**: ECAM を使うには、ACPI（*Advanced Configuration and Power Interface*）の MCFG テーブルを解析して PCIe 設定空間が物理メモリのどこにマップされているかを取得する必要があります。しかし、第6章時点の MikanOS には ACPI パーサーもメモリマネージャもまだありません。
2. **完全な下位互換性**: Mechanism #1 は、1992 年以降のほぼすべての x86 PC（実機および QEMU/VMware/VirtualBox）で、OS がどのような状態であっても BIOS 起動直後から 100% 確実に動作します。

### 10.2 PCIe ECAM の仕組みと優位性
OS の開発が進み、ACPI や 4階層ページングが整った本格的な OS（Linux や Windows）では、ECAM が標準となります。
ECAM では、物理メモリ上に **256 MiB** の巨大なアドレス領域が予約され、以下のアドレス計算式で直接メモリアクセス（MMIO）できます：

$$\mathbf{ECAM\_Address} = \text{Base} + (\text{bus} \ll 20) + (\text{device} \ll 15) + (\text{function} \ll 12) + \text{reg\_offset}$$

| 項目 | Configuration Mechanism #1 | PCIe ECAM |
|---|---|---|
| **アクセス方式** | I/O ポート `0x0CF8` / `0x0CFC` | 通常のメモリアクセス (MMIO) |
| **アドレス空間サイズ** | 1ファンクションあたり 256 バイト | 1ファンクションあたり **4,096 バイト** (拡張PCIe空間対応) |
| **マルチコア並行性** | **低い**（共有 I/O ポートへの排他ロックが必須） | **極めて高い**（コアごとに独立したメモリアドレスを読み書き可能） |
| **アクセス速度** | 低速（I/O バスサイクルによる遅延） | 高速（通常のメモリキャッシュ/バスインターコネクト） |

## 11. 第6章 6.3節の総括と次節（6.4 ポーリングでマウス入力）への展望

### 11.1 本節で達成したことの総括
1. **周辺機器探索の必然性の解明**: マザーボードごとに異なるデバイス接続位置を OS が自律的に検出するプラグ・アンド・プレイの基本思想を理解しました。
2. **I/O ポート空間と System V AMD64 ABI の完全制覇**: `IoIn32` / `IoOut32` による x86 特有の I/O ポート読み書き機構をアセンブリから実装しました。
3. **`CONFIG_ADDRESS` (0x0CF8) の数理導出**: バス・デバイス・ファンクション・レジスタオフセットを 1 つの 32 ビット整数にまとめる `MakeAddress()` のビット演算を導出しました。
4. **Header Type に基づく再帰走査アルゴリズム**: 単一ファンクションによる無駄走査の排除、および PCI-to-PCI ブリッジ（Header Type 01h）検出時のセカンダリバスへの再帰呼び出しを完成させました。
5. **xHCI USB 3.0 コントローラの特定**: Class Code `0x0C0330` を用いて、マシン上の多数の PCI デバイスから xHCI をピンポイントで特定することに成功しました。

### 11.2 次節 6.4「ポーリングでマウス入力 (`osbook_day06c`)」への展望
PCI デバイスの探索により、ついに **xHCI ホストコントローラの物理的な存在場所（Bus 0, Device 4, Function 0）** を特定することができました！

次節 **6.4節** では、以下の OS 自作のクライマックスへと進みます：
- **BAR0 (Base Address Register 0) の読み出し**: xHCI の MMIO レジスタがマップされている物理メモリベースアドレスを取得
- **xHCI ドライバの初期化**: 前節（6.2節）で学んだ DCBAA、Command Ring、Event Ring を割り当ててホストコントローラを起動
- **USB HID マウスドライバの登録**: マウスから送信されるレポート（相対移動量 $dx, dy$ とボタン押下状態）をポーリングで受信
- **マウスカーソルの動的移動**: 物理マウスを動かすと、画面上のマウスカーソルが滑らかに追従して動く感動の瞬間を迎えます！